# Pandas Revision Cheat Sheet for AI/ML

A runnable revision notebook for data loading, inspection, cleaning, transformation, aggregation, joining, time series, and ML preparation.

**Argument policy:** Every argument used in a demonstrated function is explained in an adjacent comment. Large APIs often have many optional parameters; this notebook explains every argument **shown here**, prioritizing the ones needed for AI/data work.


In [1]:
import numpy as np              # Numerical arrays and missing-value support
import pandas as pd             # Tabular data analysis
print("Pandas version:", pd.__version__)


Pandas version: 2.3.3


## 1. Series and DataFrame creation


In [2]:
# pd.Series(data, index, name, dtype)
s = pd.Series(
    data=[90, 85, 88],                 # data: values stored in the one-dimensional Series
    index=["A", "B", "C"],           # index: custom row labels; default would be 0, 1, 2
    name="score",                      # name: label for the Series; becomes a column name in a DataFrame
    dtype="int64"                      # dtype: explicit storage type
)
print(s)

df = pd.DataFrame(
    data={                              # data: dictionary keys become columns; lists become column values
        "name": ["Ana", "Ben", "Cara", "Dev"],
        "age": [24, 31, 29, 35],
        "city": ["Delhi", "Pune", "Delhi", "Mumbai"],
        "score": [88.5, 76.0, 91.0, np.nan]
    },
    index=[101, 102, 103, 104],         # index: custom row identifiers
    columns=["name", "age", "city", "score"]  # columns: sets and orders the displayed columns
)
print(df)


A    90
B    85
C    88
Name: score, dtype: int64
     name  age    city  score
101   Ana   24   Delhi   88.5
102   Ben   31    Pune   76.0
103  Cara   29   Delhi   91.0
104   Dev   35  Mumbai    NaN


## 2. Reading and writing data


In [3]:
# Typical CSV loading pattern. Uncomment when you have a file.
# df = pd.read_csv(
#     filepath_or_buffer="data.csv",   # filepath_or_buffer: file path, URL, or file-like object
#     sep=",",                         # sep: field delimiter; use "\t" for TSV
#     header=0,                         # header: row number containing column names; None means no header
#     names=None,                       # names: replacement column names; useful when header=None
#     index_col=None,                   # index_col: column name/position to use as row index
#     usecols=["age", "score"],        # usecols: load only selected columns to reduce memory
#     dtype={"age": "int64"},          # dtype: requested type per column
#     na_values=["NA", "?"],           # na_values: extra text tokens treated as missing
#     parse_dates=None,                 # parse_dates: columns to parse as datetime values
#     nrows=None,                       # nrows: maximum rows to read; useful for previews
#     encoding="utf-8"                 # encoding: character encoding used by the text file
# )

# df.to_csv(
#     path_or_buf="clean_data.csv",    # path_or_buf: destination file path or buffer
#     index=False,                      # index: False prevents row index from being written as a column
#     columns=None,                     # columns: optional subset and order of columns to export
#     sep=",",                         # sep: output delimiter
#     na_rep="",                       # na_rep: text written for missing values
#     encoding="utf-8"                 # encoding: output character encoding
# )

# Excel equivalents:
# pd.read_excel("data.xlsx", sheet_name=0)     # sheet_name: sheet name or zero-based position
# df.to_excel("output.xlsx", index=False)      # index: whether to write row labels


## 3. Inspecting data


In [4]:
print(df.head(n=2))               # n: number of first rows to return; default is 5
print(df.tail(n=2))               # n: number of last rows to return
print(df.sample(n=2, random_state=42))  # n: rows sampled; random_state: fixed seed for reproducibility
print(df.shape)                   # (rows, columns); property, not a function
print(df.columns)                 # column labels
print(df.index)                   # row labels
print(df.dtypes)                  # dtype of each column
print(df.info(verbose=True, show_counts=True))
# verbose: True lists every column
# show_counts: True displays non-null counts
print(df.describe(include="all"))
# include: "all" summarizes numeric and nonnumeric columns; default summarizes numeric only


    name  age   city  score
101  Ana   24  Delhi   88.5
102  Ben   31   Pune   76.0
     name  age    city  score
103  Cara   29   Delhi   91.0
104   Dev   35  Mumbai    NaN
    name  age    city  score
102  Ben   31    Pune   76.0
104  Dev   35  Mumbai    NaN
(4, 4)
Index(['name', 'age', 'city', 'score'], dtype='object')
Index([101, 102, 103, 104], dtype='int64')
name      object
age        int64
city      object
score    float64
dtype: object
<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, 101 to 104
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    4 non-null      object 
 1   age     4 non-null      int64  
 2   city    4 non-null      object 
 3   score   3 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 128.0+ bytes
None
       name        age   city      score
count     4   4.000000      4   3.000000
unique    4        NaN      3        NaN
top     Ana        NaN  Delhi      

## 4. Selecting rows and columns


In [5]:
print(df["score"])                # Select one column as a Series
print(df[["name", "score"]])      # Select multiple columns as a DataFrame

print(df.loc[101, "name"])        # loc[row_label, column_label]: label-based selection
print(df.loc[101:103, ["name", "score"]])  # label slice includes both endpoints
print(df.iloc[0, 0])              # iloc[row_position, column_position]: integer-position selection
print(df.iloc[0:2, 0:3])          # position slice excludes the stop position

print(df.at[101, "score"])        # at[row_label, column_label]: fast single value by label
print(df.iat[0, 3])               # iat[row_position, column_position]: fast single value by position


101    88.5
102    76.0
103    91.0
104     NaN
Name: score, dtype: float64
     name  score
101   Ana   88.5
102   Ben   76.0
103  Cara   91.0
104   Dev    NaN
Ana
     name  score
101   Ana   88.5
102   Ben   76.0
103  Cara   91.0
Ana
    name  age   city
101  Ana   24  Delhi
102  Ben   31   Pune
88.5
88.5


## 5. Filtering and query


In [6]:
mask = (df["age"] >= 29) & (df["city"] == "Delhi")  # Parenthesize each condition; & is element-wise AND
print(df.loc[mask])

print(df["city"].isin(["Delhi", "Pune"]))
# values: collection of accepted values; returns one Boolean per row

print(df["score"].between(80, 95, inclusive="both"))
# left=80: lower boundary
# right=95: upper boundary
# inclusive="both": include both boundaries; also supports "left", "right", "neither"

print(df.query("age >= @minimum_age", local_dict={"minimum_age": 29}))
# expr: string expression evaluated against column names
# local_dict: variables available in the expression; @ accesses Python-side variables


     name  age   city  score
103  Cara   29  Delhi   91.0
101     True
102     True
103     True
104    False
Name: city, dtype: bool
101     True
102    False
103     True
104    False
Name: score, dtype: bool
     name  age    city  score
102   Ben   31    Pune   76.0
103  Cara   29   Delhi   91.0
104   Dev   35  Mumbai    NaN


## 6. Missing values


In [7]:
print(df.isna())                   # True where values are missing
print(df.notna())                 # True where values are present

filled = df.fillna(
    value={"score": df["score"].mean()},  # value: scalar/dict/Series used to replace missing values
    inplace=False                         # inplace: False returns a new DataFrame instead of modifying df
)
print(filled)

print(df.dropna(
    axis=0,                       # axis=0 drops rows; axis=1 drops columns
    how="any",                   # how="any" drops if any selected value is missing; "all" requires all missing
    subset=["score"],            # subset: only inspect these columns when deciding what to drop
    thresh=None,                  # thresh: minimum number of non-missing values required; incompatible with how
    inplace=False                # inplace: return a new object
))

print(df["score"].interpolate(method="linear", limit_direction="both"))
# method="linear": estimate using linear spacing
# limit_direction="both": fill missing values in forward and backward directions


      name    age   city  score
101  False  False  False  False
102  False  False  False  False
103  False  False  False  False
104  False  False  False   True
     name   age  city  score
101  True  True  True   True
102  True  True  True   True
103  True  True  True   True
104  True  True  True  False
     name  age    city      score
101   Ana   24   Delhi  88.500000
102   Ben   31    Pune  76.000000
103  Cara   29   Delhi  91.000000
104   Dev   35  Mumbai  85.166667


Traceback (most recent call last):
  File "<exec>", line 10, in <module>
TypeError: You cannot set both the how and thresh arguments at the same time.


## 7. Duplicate data


In [8]:
dupes = pd.DataFrame({"id":[1,1,2], "value":[10,10,20]})
print(dupes.duplicated(subset=["id", "value"], keep="first"))
# subset: columns used to identify duplicates
# keep="first": mark later duplicates True; "last" keeps last; False marks every duplicate

print(dupes.drop_duplicates(subset=["id"], keep="first", ignore_index=True))
# subset: columns defining duplicate groups
# keep: occurrence retained
# ignore_index=True: rebuild index as 0..n-1


0    False
1     True
2    False
dtype: bool
   id  value
0   1     10
1   2     20


## 8. Sorting, renaming, and index management


In [9]:
print(df.sort_values(
    by=["city", "score"],         # by: column or ordered list of sort keys
    axis=0,                       # axis=0 sorts rows using column values
    ascending=[True, False],      # ascending: direction for each key
    na_position="last",           # na_position: place missing values first or last
    kind="stable",                # kind: stable preserves original order among equal keys
    ignore_index=False            # ignore_index: retain original row labels
))

print(df.rename(
    columns={"score": "exam_score"},  # columns: old-to-new column-name mapping
    index={101: "student_101"},        # index: old-to-new row-label mapping
    inplace=False                      # return renamed copy
))

reset = df.reset_index(drop=False, names="student_id")
# drop=False: preserve old index as a column
# names: name assigned to the preserved index column
print(reset)

print(reset.set_index(keys="student_id", drop=True, verify_integrity=True))
# keys: column(s) used as new index
# drop=True: remove key column after moving it to the index
# verify_integrity=True: raise an error if index values are duplicated


     name  age    city  score
103  Cara   29   Delhi   91.0
101   Ana   24   Delhi   88.5
104   Dev   35  Mumbai    NaN
102   Ben   31    Pune   76.0
             name  age    city  exam_score
student_101   Ana   24   Delhi        88.5
102           Ben   31    Pune        76.0
103          Cara   29   Delhi        91.0
104           Dev   35  Mumbai         NaN
   student_id  name  age    city  score
0         101   Ana   24   Delhi   88.5
1         102   Ben   31    Pune   76.0
2         103  Cara   29   Delhi   91.0
3         104   Dev   35  Mumbai    NaN
            name  age    city  score
student_id                          
101          Ana   24   Delhi   88.5
102          Ben   31    Pune   76.0
103         Cara   29   Delhi   91.0
104          Dev   35  Mumbai    NaN


## 9. Type conversion and text operations


In [10]:
text_df = pd.DataFrame({"number":["10","20","bad"], "name":[" Alice ","BOB","cara"]})
print(pd.to_numeric(text_df["number"], errors="coerce", downcast=None))
# arg: values to convert
# errors="coerce": invalid values become NaN; "raise" throws an error
# downcast=None: do not attempt a smaller numeric dtype

print(text_df["name"].str.strip())            # strip(): remove surrounding whitespace
print(text_df["name"].str.lower())            # lower(): lowercase strings
print(text_df["name"].str.contains("a", case=False, na=False, regex=True))
# pat="a": searched pattern
# case=False: case-insensitive matching
# na=False: missing strings produce False
# regex=True: interpret pattern as regular expression


0    10.0
1    20.0
2     NaN
Name: number, dtype: float64
0    Alice
1      BOB
2     cara
Name: name, dtype: object
0     alice 
1        bob
2       cara
Name: name, dtype: object
0     True
1    False
2     True
Name: name, dtype: bool


## 10. apply, map, assign, and replace


In [11]:
print(df["age"].map(lambda x: "30+" if x >= 30 else "under 30", na_action=None))
# arg: function or mapping applied to each Series value
# na_action=None: pass missing values to the function; "ignore" skips them

print(df.apply(lambda col: col.nunique(dropna=True), axis=0, result_type=None))
# func: function applied to each row or column
# axis=0: apply to columns; axis=1 applies to rows
# result_type: controls expansion/broadcasting for row-wise outputs; None infers behavior

assigned = df.assign(
    age_squared=lambda frame: frame["age"] ** 2,  # assign: keyword becomes new column; callable receives DataFrame
    passed=lambda frame: frame["score"].ge(80)   # ge(80): element-wise >= 80
)
print(assigned)

print(df.replace(to_replace={"city":{"Delhi":"New Delhi"}}, value=None, regex=False))
# to_replace: values or nested mapping to replace
# value=None: replacements are taken from the mapping
# regex=False: treat keys as literal values, not regex patterns


101    under 30
102         30+
103    under 30
104         30+
Name: age, dtype: object
name     4
age      4
city     3
score    3
dtype: int64
     name  age    city  score  age_squared  passed
101   Ana   24   Delhi   88.5          576    True
102   Ben   31    Pune   76.0          961   False
103  Cara   29   Delhi   91.0          841    True
104   Dev   35  Mumbai    NaN         1225   False


Traceback (most recent call last):
  File "<exec>", line 16, in <module>
ValueError: Series.replace cannot use dict-like to_replace and non-None value


## 11. GroupBy and aggregation


In [12]:
sales = pd.DataFrame({
    "region":["North","North","South","South"],
    "product":["A","B","A","B"],
    "revenue":[100,150,120,180],
    "units":[5,6,4,9]
})

grouped = sales.groupby(
    by="region",                   # by: column, list, mapping, or function defining groups
    axis=0,                        # axis=0 groups rows; axis parameter is legacy in newer pandas
    as_index=False,                # as_index=False keeps group keys as normal columns
    sort=True,                     # sort=True sorts group keys
    dropna=True,                   # dropna=True excludes missing group keys
    observed=False                 # observed: for categorical keys, include only observed categories when True
)
print(grouped.agg(
    total_revenue=("revenue", "sum"),  # Named aggregation: output=(source column, aggregation)
    average_units=("units", "mean"),
    order_count=("product", "size")
))

print(sales.groupby("region")["revenue"].transform("mean"))
# transform(func): returns group calculation aligned back to every original row


  region  total_revenue  average_units  order_count
0  North            250            5.5            2
1  South            300            6.5            2
0    125.0
1    125.0
2    150.0
3    150.0
Name: revenue, dtype: float64


<exec>:8: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.


## 12. Merge, join, and concatenate


In [13]:
customers = pd.DataFrame({"customer_id":[1,2,3], "name":["A","B","C"]})
orders = pd.DataFrame({"customer_id":[1,1,2], "amount":[50,75,100]})

print(pd.merge(
    left=customers,                 # left: left DataFrame
    right=orders,                   # right: right DataFrame
    how="left",                    # how: "left", "right", "inner", "outer", or "cross"
    on="customer_id",              # on: common join key; use left_on/right_on for differently named keys
    suffixes=("_customer", "_order"), # suffixes: labels added to overlapping non-key columns
    indicator=True,                # indicator=True adds _merge showing row origin
    validate="one_to_many"         # validate: check expected key relationship
))

print(pd.concat(
    objs=[customers, customers],    # objs: sequence or mapping of pandas objects
    axis=0,                         # axis=0 adds rows; axis=1 adds columns
    join="outer",                  # join: align other axis using outer union or inner intersection
    ignore_index=True,             # ignore_index=True builds a fresh sequential index
    keys=None,                     # keys: optional outer hierarchical index labels
    sort=False,                    # sort: whether to sort non-concatenation axis
    verify_integrity=False         # verify_integrity: check for duplicate labels on concatenation axis
))


   customer_id name  amount     _merge
0            1    A    50.0       both
1            1    A    75.0       both
2            2    B   100.0       both
3            3    C     NaN  left_only
   customer_id name
0            1    A
1            2    B
2            3    C
3            1    A
4            2    B
5            3    C


## 13. Pivoting, melting, and crosstabs


In [14]:
print(sales.pivot_table(
    values="revenue",              # values: column whose values are aggregated
    index="region",                # index: row grouping key(s)
    columns="product",             # columns: column grouping key(s)
    aggfunc="sum",                 # aggfunc: aggregation function or list/dict of functions
    fill_value=0,                  # fill_value: replace missing result cells
    margins=True,                  # margins=True adds total row/column
    margins_name="Total",          # margins_name: label for totals
    observed=False,                # observed: categorical grouping behavior
    sort=True                      # sort: sort group keys
))

wide = pd.DataFrame({"id":[1,2], "math":[80,90], "science":[85,88]})
print(wide.melt(
    id_vars=["id"],                # id_vars: columns retained as identifiers
    value_vars=["math","science"],# value_vars: columns unpivoted into rows
    var_name="subject",            # var_name: name for former column labels
    value_name="score",            # value_name: name for collected cell values
    ignore_index=True              # ignore_index: create fresh index
))

print(pd.crosstab(
    index=sales["region"],          # index: values defining result rows
    columns=sales["product"],       # columns: values defining result columns
    values=None,                   # values: optional values to aggregate; None counts occurrences
    aggfunc=None,                  # aggfunc: required when values is provided
    normalize=False,               # normalize: False gives counts; "index" gives row proportions
    margins=True                   # margins=True adds totals
))


product    A    B  Total
region                  
North    100  150    250
South    120  180    300
Total    220  330    550
   id  subject  score
0   1     math     80
1   2     math     90
2   1  science     85
3   2  science     88
product  A  B  All
region            
North    1  1    2
South    1  1    2
All      2  2    4


## 14. Date and time operations


In [15]:
dates = pd.Series(["2026-01-15", "2026-02-20", "invalid"])
parsed = pd.to_datetime(
    arg=dates,                     # arg: scalar, array-like, Series, or DataFrame to parse
    errors="coerce",              # errors="coerce": invalid dates become NaT
    format="%Y-%m-%d",            # format: exact datetime format for predictable parsing
    dayfirst=False,                # dayfirst: prefer day-month order when format is not fixed
    yearfirst=True,                # yearfirst: prefer year-first interpretation
    utc=False                     # utc=True returns timezone-aware UTC values
)
print(parsed)
print(parsed.dt.year)              # Extract year component
print(parsed.dt.month_name())      # Convert month number to month name

indexed = pd.DataFrame({"value":[10,20,30]}, index=pd.date_range(
    start="2026-01-01",           # start: first timestamp
    periods=3,                    # periods: number of timestamps generated
    freq="D",                     # freq: D=daily; H=hourly; MS=month-start, etc.
    tz=None                       # tz: optional timezone
))
print(indexed.resample("2D", closed="left", label="left").sum())
# rule="2D": build two-day bins
# closed="left": include left bin boundary
# label="left": label each bin with its left boundary


0   2026-01-15
1   2026-02-20
2          NaT
dtype: datetime64[ns]
0    2026.0
1    2026.0
2       NaN
dtype: float64
0     January
1    February
2         NaN
dtype: object
            value
2026-01-01     30
2026-01-03     30


## 15. Window operations


In [16]:
series = pd.Series([1,2,3,4,5])
print(series.rolling(
    window=3,                     # window: number of observations or time offset in each window
    min_periods=1,                # min_periods: required observations before producing a value
    center=False,                 # center=False labels result at the right edge
    closed=None                   # closed: endpoint inclusion for offset windows
).mean())

print(series.expanding(min_periods=1).mean())
# min_periods: minimum observations before calculating expanding result

print(series.ewm(span=3, adjust=True, min_periods=0).mean())
# span: decay specification; larger span means slower decay
# adjust=True: use normalized weights over complete history
# min_periods: minimum observations before output


0    1.0
1    1.5
2    2.0
3    3.0
4    4.0
dtype: float64
0    1.0
1    1.5
2    2.0
3    2.5
4    3.0
dtype: float64
0    1.000000
1    1.666667
2    2.428571
3    3.266667
4    4.161290
dtype: float64


## 16. Memory, categories, and safe assignment


In [17]:
large_labels = pd.Series(["red","blue","red","green"])
category_labels = large_labels.astype("category")  # category stores repeated labels using integer codes
print(category_labels.cat.categories)               # Unique allowed category labels
print(category_labels.cat.codes)                    # Integer code for each value

safe = df.copy(deep=True)          # deep=True copies data and index; avoids accidental shared modifications
safe.loc[safe["age"] > 30, "senior"] = True  # Use .loc for conditional assignment
safe["senior"] = safe["senior"].fillna(False).astype(bool)
print(safe)
print(safe.memory_usage(index=True, deep=True))
# index=True: include index memory
# deep=True: inspect object/string contents more deeply


Index(['blue', 'green', 'red'], dtype='object')
0    2
1    0
2    2
3    1
dtype: int8
     name  age    city  score  senior
101   Ana   24   Delhi   88.5   False
102   Ben   31    Pune   76.0    True
103  Cara   29   Delhi   91.0   False
104   Dev   35  Mumbai    NaN    True
Index     160
name      113
age        32
city      120
score      32
senior      4
dtype: int64


<exec>:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


## 17. ML preparation patterns


In [18]:
ml = pd.DataFrame({
    "age":[20,30,np.nan,40],
    "city":["Delhi","Pune","Delhi","Mumbai"],
    "target":[0,1,0,1]
})
ml["age"] = ml["age"].fillna(ml["age"].median())  # Median imputation for numeric feature
encoded = pd.get_dummies(
    data=ml,                       # data: Series/DataFrame to encode
    columns=["city"],              # columns: categorical columns transformed into indicators
    prefix=None,                  # prefix: optional prefix for generated column names
    prefix_sep="_",               # prefix_sep: separator between original name and category
    dummy_na=False,               # dummy_na=True creates an explicit missing-value indicator
    drop_first=False,             # drop_first=True removes first level; useful in some linear models
    dtype="int8"                  # dtype: storage type for indicator columns
)
print(encoded)

X = encoded.drop(columns=["target"], errors="raise")
# labels/columns: drop selected column labels
# errors="raise": fail when a requested label is missing; "ignore" silently skips
y = encoded["target"].to_numpy(dtype=np.int64, copy=True)
# dtype: requested NumPy dtype
# copy=True: force an independent NumPy array
print(X.to_numpy(dtype=np.float32, copy=True), y)


    age  target  city_Delhi  city_Mumbai  city_Pune
0  20.0       0           1            0          0
1  30.0       1           0            0          1
2  30.0       0           1            0          0
3  40.0       1           0            1          0
[[20.  1.  0.  0.]
 [30.  0.  0.  1.]
 [30.  1.  0.  0.]
 [40.  0.  1.  0.]] [0 1 0 1]


## Final Pandas checklist

Master: construction, I/O, inspection, selection, Boolean filtering, missing data, duplicates, sorting, type conversion, string/date accessors, `groupby`, merge/concat, pivot/melt, window operations, categories, safe assignment, and ML encoding.

**Most important mental model:** rows are observations, columns are features, the index labels rows, and vectorized column operations should replace manual Python loops whenever possible.
